In [12]:
!wget -O sparql-anything.jar https://github.com/SPARQL-Anything/sparql.anything/releases/download/v1.1.0/sparql-anything-v1.1.0.jar
!java -version

--2026-02-03 08:07:24--  https://github.com/SPARQL-Anything/sparql.anything/releases/download/v1.1.0/sparql-anything-v1.1.0.jar
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/303967701/d4b57144-968f-4781-a18e-ead6b6eb0da1?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-02-03T08%3A42%3A12Z&rscd=attachment%3B+filename%3Dsparql-anything-v1.1.0.jar&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-02-03T07%3A42%3A10Z&ske=2026-02-03T08%3A42%3A12Z&sks=b&skv=2018-11-09&sig=vU95LJCXYcVVNEirseT1oShWLsu35ZiqxzuxfsX6GaA%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MDEwOTU2NSwibmJmIjoxNzcwMTA1OTY1LCJwYXRoIjoi

In [ ]:
import os
sparql_query = """
PREFIX xyz: <http://sparql.xyz/facade-x/data/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX : <http://www.semanticweb.org/ModSem/SportInvernaliOlimpici/>

CONSTRUCT {
    ?atletaIRI a :AtletaInvernale ;
               :haNome ?nomeStr ;
               :haEta ?etaInt ;
               :haPunteggio ?punteggioInt ;
               :praticaSport ?sportIRI ;
               :partecipaAGara ?garaIRI .

    ?garaIRI a :GaraOlimpica ;
             :siSvolgeIn ?luogoIRI ;
             :garaDi ?sportIRI ;
             :haData ?dataDate .

    ?atletaIRI :vinceMedaglia ?medagliaIRI .

    ?medagliaIRI a :Medaglia ;
                 :haTipo ?tipoMedagliaIRI ;
                 :assegnataInGara ?garaIRI .
}
WHERE {
    # Specifica del delimitatore
    SERVICE <x-sparql-anything:location=file:///content/Dati_Gare_Invernali.csv,csv.headers=true,csv.delimiter=;> {

        ?row xyz:Nome ?nomeStr ;
             xyz:Eta ?etaStr ;
             xyz:Punteggio ?punteggioStr ;
             xyz:Sport_Praticato ?sportStr ;
             xyz:Gara_ID ?garaIdStr ;
             xyz:Luogo_Gara ?luogoStr .

        OPTIONAL { ?row xyz:Data_Gara ?dataRaw . }
        OPTIONAL { ?row xyz:Medaglia_Tipo ?medagliaTipoStr . }

        # URI Atleta
        BIND(IRI(CONCAT(STR(:), ENCODE_FOR_URI(?nomeStr))) AS ?atletaIRI)

        # URI Sport e Luogo
        BIND(IRI(CONCAT(STR(:), ?sportStr)) AS ?sportIRI)
        BIND(IRI(CONCAT(STR(:), ENCODE_FOR_URI(?luogoStr))) AS ?luogoIRI)

        # URI Gara
        BIND(IRI(CONCAT(STR(:), ?garaIdStr)) AS ?garaIRI)

        # Casting Numeri
        BIND(xsd:integer(?etaStr) AS ?etaInt)
        BIND(xsd:integer(?punteggioStr) AS ?punteggioInt)

        # Conversione data: da 10/02/2022 a 2022-02-10
        BIND(IF(BOUND(?dataRaw),
             xsd:date(CONCAT(SUBSTR(?dataRaw, 7, 4), "-", SUBSTR(?dataRaw, 4, 2), "-", SUBSTR(?dataRaw, 1, 2))),
             ?null) AS ?dataDate)

        # Filtro Medaglie (Gestione celle vuote o "NULL")
        FILTER (BOUND(?medagliaTipoStr) && ?medagliaTipoStr != "NULL" && ?medagliaTipoStr != "")

        # URI Medaglia
        BIND(IRI(CONCAT(STR(:), "Medaglia_", ?medagliaTipoStr, "_", ?garaIdStr)) AS ?medagliaIRI)
        BIND(IRI(CONCAT(STR(:), ?medagliaTipoStr)) AS ?tipoMedagliaIRI)
    }
}
"""

with open('mapping.sparql', 'w') as f:
    f.write(sparql_query)
print("Query SPARQL creata")

csv_path = "/content/Dati_Gare_Invernali.csv"

if os.path.exists(csv_path):
    print("Esecuzione SPARQL Anything...")
    # Esegue il jar
    !java -jar sparql-anything.jar -q mapping.sparql -o output.ttl

    # Verifica output
    if os.path.exists("output.ttl") and os.path.getsize("output.ttl") > 0:
        print("\n Il file 'output.ttl' contiene i dati.")
    else:
        print("error: Verifica il contenuto del CSV.")
else:
    print(f" error: Non trovo il file {csv_path}.")

Query SPARQL creata
Esecuzione SPARQL Anything...

 Il file 'output.ttl' contiene i dati.
